# Lab 2: GPT from scratch

In this lab, you will dive into the inner workings of the GPT architecture. You will walk through a complete implementation of the architecture in PyTorch, instantiate this implementation with pre-trained weights, and put the resulting model to the test by generating text. At the end of this lab, you will understand the building blocks of the GPT architecture and how they are connected.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

In [2]:
from dataclasses import dataclass

import torch
import torch.nn as nn

## Part 1: GPT architecture

GPT-2 was first described by [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). To faithfully implement the model, one needs to also read the earlier paper by [Radford et al. (2018)](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf). Another important source of information is the code released by OpenAI, which is available on GitHub ([link](https://github.com/openai/gpt-2)).

The GPT architecture is made up of a stack of Transformer blocks. Each block has two main parts: one handles multi-head self-attention, and the other is a feed-forward network. Before these parts do their work, their input undergoes layer normalisation, and residual connections are added to help the model learn more effectively. The input to the architecture is a sequence of token IDs; these are turned into embeddings and augmented with information about the absolute position of each token in the sequence. The output layer converts the internal representations into logit scores for every token in the vocabulary.

### Model configuration

[Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) present four increasingly larger GPT models based on the same architecture. Here, we will implement the smallest of these, characterised by the following hyperparameters:

In [3]:
@dataclass
class Config:
    n_vocab: int = 50_257 #Size of the dictionary
    n_ctx: int = 1024 # Max sequence length (memory), after this the model does not remember the 1st sequence
    n_embd: int = 768 # Dimension of the vector (The "width" of the diagram)
    n_head: int = 12 # Number of attention head
    n_layer: int = 12 # Number of transformer blocks 

#### 🧩 Task 2.01: Model configuration

Explain the purpose of these hyperparameters. In particular, where does the number 50,257 come from?

ANSWER: Hyperparameters are external configurations of and vaibles set by data scientics before training a machine learning model and acts as a structural blueprint. These parameters are manually tuned to optimize performence, such as learning rate, number of hidden layers, or batch size. 
The number 50_257 is the total size of the dictionary containing the unique tokes that the model can see. This numbers comes from running BPE algorithm on OpenAi for GPT-2 dataset to find the most common sequences of characters (sub-words). 50 000 is the number of the merges and the math is -> 256 + 50 000 = 50 257.

### GELU activation function

We start by implementing the feed-forward network. This is a standard two-layer network with a Gaussian Error Linear Unit (GELU) activation function ([Hendrycks and Gimpel, 2016](https://doi.org/10.48550/arXiv.1606.08415)).

The GELU is a smooth version of the rectified linear unit (ReLU) that weights inputs by their value under the cumulative distribution function of the standard Gaussian. This function is commonly denoted by $\Phi$. For example, $\text{GELU}(0{.}5) = 0{.}5 \cdot \Phi(0{.}5) \approx 0{.}5 \cdot 0{.}6915 = 0{.}3457$ because approximately 69.15% of normally distributed data lies to the left of $0{.}5$.

When GPT-2 was released, computing the GELU exactly was expensive, and the released code therefore used an approximation originally presented by [Page (1977)](https://doi.org/10.2307/2346872). We follow suit here, as we want to create a replica of the original model. However, it is worth mentioning that PyTorch now offers an exact implementation of the GELU so fast that using an approximation is unnecessary.

In [4]:
def gelu(x):
    return 0.5 * x * (1 + torch.tanh((2 / torch.pi) ** 0.5 * (x + 0.044715 * x**3)))

#### 🎓 Task 2.02: Mathematical properties of the GELU

Find the minimal output value of the (approximated) GELU and the input value for which it yields that output. Use a service such as [WolframAlpha](https://www.wolframalpha.com/) for the necessary derivations. What are the main differences between the GELU and the ReLU?

ANSWER: The minimal output = -0.170041       and the minimal input is at x=-752461 
The main defferences: 1) Both are non-linear functions used in neural networks, but their mathematical formulations differ a lot from each other.   2) ReLU has the function max(0,x), known for its simplicity and effeciency. GELU has a smother function (x.Q(x)) based on cumulative distribution.  3) ReLU is 0 for all negative inputs and linear (x) for positive inputs.  GELU has negative outputs for negative inputs, which creates a more probabilistic.


![alt text](2026-02-13_00-18.png)


![alt text](wolframalpha.png)



### Feed-forward network

Next, here is the code for the feed-forward network. Note that we follow the released code and use the name **multi-layer perceptron (MLP)** rather than “feed-forward network”.

In [5]:
class MLP(nn.Module): # MLP (Multi-Layer Perception) == FNN (feed-forward network) in the diagram
    def __init__(self, config): 
        super().__init__()
        # Expansion: 768 -> 3072
        self.c_fc = nn.Linear(config.n_embd, config.n_embd * 4)
        #Contraction: 3072 -> 768
        self.c_proj = nn.Linear(config.n_embd * 4, config.n_embd)

    def forward(self, x):
        batch_size, seq_len, n_embd = x.shape # x shape: [batch_size, seq_len, 768]
        x = self.c_fc(x) # Projects up to 3072
        x = gelu(x) # Applies the activation function
        x = self.c_proj(x)  # Projects back to 768
        return x

#### 🎓 Task 2.03: Shape annotations

One of the most common errors in deep learning is a mismatch in tensor dimensions. To avoid this, it is good practice to annotate PyTorch code with shapes. For example, suppose you are given the following code:

In [6]:
f = nn.Linear(5, 7)
x = torch.rand(2, 3, 5)
y = f(x)

The annotation of this code with shapes would look as follows:

In [7]:
f = nn.Linear(5, 7)
# not a tensor variable; needs no annotation

x = torch.rand(2, 3, 5)
# shape of x: [2, 3, 5]

y = f(x)
# shape of y: [2, 3, 7]

Annotate the shapes in the `forward()` method of the feed-forward network. Instead of using actual numbers, refer to dimension sizes by symbolic names such as `n_embd`, `batch_size` (number of samples in a batch of input data) and `seq_len` (length of an input sequence). You can introduce additional names and other notation you find useful. Make your annotations as detailed as you need them to explain how the shapes change from one line to the next.

ANSWER: take a look at the output for the clases: "class Config_example" and "class annotate_forward(nn.Module)"

In [8]:
@dataclass
class Config_example:
    n_embd_example: int = 8
    batch_size: int = 1
    seq_len: int = 5


In [9]:
class annotate_forward(nn.Module):
    def __init__(self, config_example):
        super().__init__()
        self.c_fc = nn.Linear(config_example.n_embd_example, config_example.n_embd_example * 4)
        self.c_proj = nn.Linear(config_example.n_embd_example *4 , config_example.n_embd_example)

        print(f"-- Initialization --")
        print(f"Expansion matrix size: {self.c_fc.weight.shape}")
        print(f"Contraction Matrix size: {self.c_proj.weight.shape}")

    def forward(self, x):
        print(f"\n-- Forward pass (processing data)--")
        print(f"fStep A (Input):          {x.shape}  <-- Start with size 8")
        x = self.c_fc(x)
        print(f"Step B (Expanded):       {x.shape} <-- Expanded to 32!")
        x = self.c_proj(x)
        print(f"Step C (Projected):      {x.shape}  <-- Back to size 8")
        return x

config = Config_example
model = annotate_forward(config)

dummy_input = torch.rand(config.batch_size, config.seq_len, config.n_embd_example)
output = model(dummy_input)


-- Initialization --
Expansion matrix size: torch.Size([32, 8])
Contraction Matrix size: torch.Size([8, 32])

-- Forward pass (processing data)--
fStep A (Input):          torch.Size([1, 5, 8])  <-- Start with size 8
Step B (Expanded):       torch.Size([1, 5, 32]) <-- Expanded to 32!
Step C (Projected):      torch.Size([1, 5, 8])  <-- Back to size 8


### Causal mask

Our next goal is to implement the core of the GPT architecture: the multi-head attention mechanism.

Recall that the attention mechanism in the Transformer decoder must be restricted to attending only to previously generated tokens. This type of attention is also called **causal attention**. In practice, we implement it through a masking technique that sets the post-softmax attention weights of future tokens to zero. The following utility function implements such a mask:

In [10]:
def make_causal_mask(n):
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)

#### 🧩 Task 2.04: Causal mask

Have a close look at the following code and run it to see the result. What are the shapes of `x` and `mask`? Given that the shapes are different, why does the addition operation in the last line not raise an error? What is the shape of the result?

How does the addition operation implement masking? (Recall that the attention scores are normalised using the softmax function.)

ANSWER: The shape of x: 1 -> Batch size (i sentence), 2 -> Number of heads (Multi-head attention), 3 -> Sequence length (Query/Target Token),
3 -> Sequence length (Key/Source Token).

shape of Mask = ([5, 5]) which is created by the function make_causal_mask().

Firstly, mask[:3, :3], cuts the 5x5 mask down to a 3x3 matrix to match the current sequence length. 
Secondly, 
                        

In [11]:
x = torch.rand(1, 2, 3, 3)
mask = make_causal_mask(5)
x + mask[:3, :3]

tensor([[[[0.3232,   -inf,   -inf],
          [0.5170, 0.3312,   -inf],
          [0.9657, 0.0504, 0.1566]],

         [[0.1384,   -inf,   -inf],
          [0.0557, 0.7372,   -inf],
          [0.8615, 0.9576, 0.8533]]]])

### Attention mechanism

Here is the code for the multi-head attention mechanism:

In [12]:
class Attention(nn.Module): # the implementation of Masked Multi-Head Attention, which allows the model to look back at priviuos words to understand the the context
    def __init__(self, config):
        super().__init__()

        assert config.n_embd % config.n_head == 0  
        self.n_head = config.n_head
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.register_buffer("mask", make_causal_mask(config.n_ctx), persistent=False)

    def forward(self, x):
        print(f"\n----- ATTENTION LAYER START ----")
        print(f"Input X:   {x.shape} (Batch, seq, Emb)")
        batch_size, seq_len, n_embd = x.shape
        head_embd = n_embd // self.n_head
        q, k, v = self.c_attn(x).chunk(3, dim=-1)
 
        print(f"\n---Step 1: Q, K, V Extraction ---")
        print(f"Q (Query): {q.shape} (Batch, seq, Emb)")
        print(f"K (Keys): {k.shape}")
        print(f"V (Values): {v.shape}")
        q = q.view(batch_size, seq_len, self.n_head, head_embd)
        k = k.view(batch_size, seq_len, self.n_head, head_embd)
        v = v.view(batch_size, seq_len, self.n_head, head_embd)
        print(f"\n---Step 2: Reshape (Split Heads)")
        print(f"Q Reshaped:   {q.shape} (Batch, Seq, Head, HeadDim)")
        print(f"K Reshaped:   {k.shape} (Batch, Seq, Head, HeadDim)")
        print(f"V Reshaped:   {v.shape} (Batch, Seq, Head, HeadDim)")
        q = q.transpose(-2, -3)
        k = k.transpose(-2, -3)
        v = v.transpose(-2, -3)
        print(f"\n---Step 3: Transpose (swap head and seq)")
        print(f"Q Transposed:   {q.shape} (Batch, Head, Seq, HeadDim)")
        print(f"K Transposed:   {k.shape} (Batch, Head, Seq, HeadDim)")
        print(f"V Transposed:   {v.shape} (Batch, Head, Seq, HeadDim)")
        x = q @ k.transpose(-1, -2)
        print(f"\n--- Step 4: Attention Scores (Q @ K.T) ---")
        print(f"Scores:  {x.shape} (Batch, Heads, Seq, Seq)")
        x = x / head_embd**0.5
        x = x + self.mask[:seq_len, :seq_len]
        print(f"\n--- Step 5: Scale and Mask ---")
        print(f"Masked:  {x.shape} (Shape unchanged, future positions are -inf)")

        x = torch.softmax(x, dim=-1)
        print(f"\n--- Step 6: Softmax ---") # Convert scores to probabilities (0.0 to 1.0)
        print(f"Softmax: {x.shape} (Probabilities calculated)")

        x = x @ v
        print(f"\n--- Step 7: Weighted Sum ---")
        print(f"Output:  {x.shape} (Batch, Heads, Seq, Head_Dim)")

        x = x.transpose(-2, -3).contiguous()        
        x = x.view(batch_size, seq_len, n_embd)   
        print(f"\n--- Step 8: Concatenate Heads ---")
        print(f"Merged:  {x.shape} (Batch, Seq, Emb_Dim) -> Back to original shape")

        x = self.c_proj(x)
        print(f"\n--- Step 9: Final Linear Projection ---")
        print(f"Final:   {x.shape}")
        return x

#### 🎓 Task 2.05: Multi-head attention

Trace the input `x` through the `forward()` method line by line and annotate the shapes of all tensor variables. Identify all lines that rely on broadcasting.

ANSWER: This code is the the implementation of Masked Multi-Head Attention, which allows the model to look back at priviuos words to understand the the context. We take the input (x) and split into Q (current token's focus), K (target token's identity) and V (content information). The mechanism calculates a weighted sum of the values (V), where the weights are determined by dot prodoct of the Query (Q) and the Key (K). We also have a causal mask function that ensures tokens only attend to past tokens, preventing future information leakage. For the track of (x), check the the print results at the end of the notebook. 

### Layer normalisation

As mentioned above, the inputs to both the feed-forward network and the multi-head attention mechanism undergo **layer normalisation**. This normalises the inputs to have zero mean and unit variance across the activations. [Ba et al. (2016)](https://doi.org/10.48550/arXiv.1607.06450) introduce two trainable parameters (called $\gamma$ and $\beta$ in the paper) that allow the network to learn an appropriate scale and shift for the normalised values.

We implement layer normalisation as follows:

In [13]:
class LayerNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.g = nn.Parameter(torch.ones(config.n_embd))
        self.b = nn.Parameter(torch.zeros(config.n_embd))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(unbiased=False, dim=-1, keepdim=True)
        return self.g * (x - mean) / torch.sqrt(variance + 1e-05) + self.b

#### 🧩 Task 2.06: Layer normalisation

What is the relevance of the `keepdim=True` keyword argument in the `mean()` and `var()` functions? What would happen if we omitted it?

What is the relevance of the constant 1e-05? What could happen if we omitted it?

ANSWER: "keepdim=True"-> ensures that the number of dimentions for the outputdata are the the same shape as the input data (x). Without "keepdim=True", the shape could be changed during the computations.
The conatant "1e-05" tells the tolerance level for the calculation. without that constant the program could crasch.


### Decoder block

We now combine the feed-forward network, the multi-head attention mechanism and the layer normalisation into a decoder block.

In [14]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config)
        self.attn = Attention(config)
        self.ln_2 = LayerNorm(config)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

#### 🎓 Task 2.07: Pre-norm and post-norm architectures

The original Transformer ([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)) is a “post-norm architecture”, where the normalisation is applied **after** each residual block. In contrast, GPT-2 is a “pre-norm architecture”, where the normalisation is applied **before**. Find the passage in Section&nbsp;2.3 of [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) that reports on this modification.

ANSWER: "Layer normalization (Ba et al., 2016)
was moved to the input of each sub-block, similar to a
pre-activation residual network (He et al., 2016) and an
additional layer normalization was added after the final self-
attention block"

[Xiong et al. (2020)](https://arxiv.org/pdf/2002.04745) compare pre-norm and post-norm architectures empirically. Read the abstract of their paper and summarise their main findings. According to these findings, what are the benefits of the pre-norm architecture?

ANSWER: Their main finding is that in an original-designed Post-LN transformer, the expected gradients of the pramaetrs are very large, which slows the training process and makes it unstable. They believe that adding a warm-up stage could practically avoid this problem. But their findings show that by removing this stage in Pre-LN, and put the layer normalization inside the residual blocks, the gradients are well-behaved. 

### Model

We now have almost all components in place to complete the implementation of the GPT-2 model. The only thing  missing are the position embeddings. These simply associate an embedding vector with every position in the context window. To set them up, we first define another utility function:

In [15]:
def make_positions(n):
    return torch.arange(n, dtype=torch.long)

We then code the complete model as follows:

In [16]:
class Model(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.n_vocab, config.n_embd)
        self.wpe = nn.Embedding(config.n_ctx, config.n_embd)
        self.h = nn.Sequential(*(Block(config) for _ in range(config.n_layer)))
        self.ln_f = LayerNorm(config)
        self.lm_head = nn.Linear(config.n_embd, config.n_vocab, bias=False)
        self.register_buffer("pos", make_positions(config.n_ctx), persistent=False)

    def forward(self, x):
        batch_size, seq_len = x.shape
        wte = self.wte(x)
        wpe = self.wpe(self.pos[:seq_len])
        x = wte + wpe
        x = self.h(x)
        x = self.ln_f(x)
        x = self.lm_head(x)
        return x

#### 🧩 Task 2.08: Buffers

Our implementation registers the vector of positions as a buffer. (Earlier, we also registered the causal mask as a buffer.) Consult the PyTorch documentation to determine the benefits of registering a tensor as a buffer, in contrast to computing it in the `forward()` method.

ANSWER: If we need the non-learnable data to accompany the model but do not want it to reacive gradients in optimization, then the usage of the register_buffer is a big benefit. In constrant if we passed it insid the forward(), it slow down the computation speed and do many unneccessarily operations.  

#### 🎓 Task 2.09: Number of trainable parameters

The model we have implemented is the smallest one presented by [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). But how many trainable parameters exactly does it have? Interestingly, the number originally reported by the authors is wrong! What number did they report?

ANSWER: The correct number the trainable parameters is 163.04M and the reported number by the paper is 117M. 

Your task is to write code to compute the number of parameters yourself. This should only take 1–3 lines of code. What number do you get when you apply this code to a fresh model instance? We get the number 163.04M parameters (check the code snippet below).

[Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) followed the original Transformers paper ([Vaswani et al., 2017](https://doi.org/10.48550/arXiv.1706.03762)) and shared the trainable weights between the token embedding and the final linear layer. Implement this weight sharing strategy. (Hint: This only requires one line of code.) Then, re-compute the number of trainable parameters for the modified model. What number do you get now? How large is the reduction caused by the weight sharing?

ANSWER: Before Sharing = 163.04M,    After Sharing = 124.44M      The reduction = 38.59M

## Part 2: Load pre-trained weights

Now that you have a complete implementation of the GPT-2 model in place, you can instantiate it by loading the pre-trained weights released by OpenAI. These weights were originally provided in the TensorFlow format. For this lab, we have re-packaged them as a single file in NumPy’s `.npz` archive format. We can load it as follows:

In [17]:
import numpy as np

pretrained = np.load("/courses/TDDE09/tdde09/labs/lab2/gpt-2-pretrained.npz", allow_pickle=True)

FileNotFoundError: [Errno 2] No such file or directory: '/courses/TDDE09/tdde09/labs/lab2/gpt-2-pretrained.npz'

The result `pretrained` is a dictionary mapping names to NumPy arrays. When you print the names, you will see that they correspond to the attributes of our network modules, even though the names differ. For example, the array `h0.attn.c_attn.b` holds the biases (`b`) of the `c_attn` linear layer in the attention mechanism (`attn`) of the first transformer block (`h0`).

#### 🎓 Task 2.10: Load pre-trained weights

Create a model from the pre-trained weights. To do this, you need to instantiate a fresh model and write the contents of each array from the `npz` archive with the pre-trained weights into the corresponding tensor. To make this a bit easier, here is a utility function that copies data from a NumPy array `source` to a PyTorch tensor `target`:

In [ ]:
def copy_weights(target: torch.Tensor, source: np.ndarray ):
    assert source.shape == target.shape
    with torch.no_grad():
        target.copy_(torch.tensor(source, dtype=torch.float32))

You can start from this skeleton code:

In [ ]:
def from_pretrained() -> Model:
    model = Model(Config())
    pretrained = np.load("/courses/TDDE09/tdde09/labs/lab2/gpt-2-pretrained.npz", allow_pickle=True)
    # TODO: Copy the weights from `pretrained` to `model`
    copy_weights(model.wpe.weight, pretrained["wpe"])
    copy_weights(model.wte.weight, pretrained["wte"])

    for i in range(len(model.h)):
        copy_weights(model.h[i].attn.c_attn.bias,
                     pretrained[f"h{i}.attn.c_attn.b"])
        copy_weights(model.h[i].attn.c_attn.weight,
                     pretrained[f"h{i}.attn.c_attn.w"].T)
        copy_weights(model.h[i].attn.c_proj.bias,
                     pretrained[f"h{i}.attn.c_proj.b"])
        copy_weights(model.h[i].attn.c_proj.weight,
                     pretrained[f"h{i}.attn.c_proj.w"].T)
        copy_weights(model.h[i].ln_1.g,
                     pretrained[f"h{i}.ln_1.g"])
        copy_weights(model.h[i].ln_1.b,
                     pretrained[f"h{i}.ln_1.b"])
        copy_weights(model.h[i].ln_2.g,
                     pretrained[f"h{i}.ln_2.g"])
        copy_weights(model.h[i].ln_2.b,
                     pretrained[f"h{i}.ln_2.b"])
        copy_weights(model.h[i].mlp.c_fc.bias,
                     pretrained[f"h{i}.mlp.c_fc.b"])
        copy_weights(model.h[i].mlp.c_fc.weight,
                     pretrained[f"h{i}.mlp.c_fc.w"].T)
        copy_weights(model.h[i].mlp.c_proj.bias,
                     pretrained[f"h{i}.mlp.c_proj.b"])
        copy_weights(model.h[i].mlp.c_proj.weight,
                     pretrained[f"h{i}.mlp.c_proj.w"].T)                  
    copy_weights(model.ln_f.g, pretrained["ln_f.g"])
    copy_weights(model.ln_f.b, pretrained["ln_f.b"])
    model.lm_head.weight = model.wte.weight

    print(f"Pretrained weights loaded successfully! {len(model.h)} blocks, {model.config.n_head} heads, {model.config.n_embd} embedding size.")
    return model
model = from_pretrained()

In [ ]:
#print(pretrained.files)
#pretrained['h0.attn.c_attn.b']
print(model.h[0].attn.c_attn.weight.shape)
print(pretrained["h0.attn.c_attn.w"].shape)
print(model.lm_head.weight.data_ptr() == model.wte.weight.data_ptr())


tot_params = sum(p.numel() for p in Model(Config()).parameters())
print(f"Total parameters in model Before sharing weights: {tot_params/1e6:.2f}M")

#model.lm_head.weight = model.wte.weight

tot_params_after_sharing = sum(p.numel() for p in model.parameters())
print(f"Total parameters in model After sharing weights: {tot_params_after_sharing/1e6:.2f}M")

**Important:** One technical detail to note is that PyTorch stores the weights of linear layers in a transposed form. For example, a linear layer created as `nn.Linear(2, 3)` has a weight matrix of shape [3, 2].

## Part 3: Put the model to use

In the third and final part of this lab, you will use the pre-trained model to generate text and evaluate it on a standard benchmark.

### Sampling-based text generation

The easiest way to generate text with a language model is by using a **greedy approach**. This method works by creating text one token at a time. At each step, the model takes the previously generated text (called the **context**) as input and adds the token with the highest output logit as a new token. The code in the next cell defines a function `generate()` that forms the core of a greedy generator:

In [ ]:
def generate(model, context, context_size=1024, n_tokens=20):
    for _ in range(n_tokens):
        context = context[:, -context_size:]
        with torch.no_grad():
            logits = model(context)[:, -1, :]
        next_idx = torch.argmax(logits, dim=-1, keepdim=True)
        context = torch.cat([context, next_idx], dim=-1)
    return context


    #-----------------------smapling based text generation------------------Task 2.11-----------------------

# def generate(model, context, context_size=1024, n_tokens=20, temperature=1.0, top_k=None):

#     for _ in range(n_tokens):
#         context_crop = context[:, -context_size:]
#         with torch.no_grad():
#             logits = model(context_crop)[:, -1, :]
#         if temperature > 0:
#             logits = logits / temperature
    
#         if top_k is not None:
#             v, _ = torch.topk(logits, top_k)
#             cutoff = v[:, [-1]]
#             logits[logits < cutoff] = float('-inf')

#         probs = torch.softmax(logits, dim=-1)
#         next_idx = torch.multinomial(probs, num_samples=1)
        
#         context = torch.cat([context, next_idx], dim=-1) 
#     return context

To use this function with an actual text input, you need a tokeniser to first encode the text into a vector of token IDs, and later decode the generated `context` into new text. The reference implementation of the GPT-2 tokeniser is in the library `tiktoken`. The code in the next cell sets up the tokeniser, loads the pretrained model from Task&nbsp;2.10, and then defines a helper function that handles the encoding and decoding.

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
model = from_pretrained()


def generate_helper(text, context_size=1024, n_tokens=20):
    context = torch.tensor([tokenizer.encode(text)], dtype=torch.long)
    context = generate(model, context, context_size=context_size, n_tokens=n_tokens)
    return tokenizer.decode(context[0].tolist())

You can use this helper function to generate text as follows:

In [ ]:
generate_helper("Linköping University is")

**Tip:** If you did not manage to complete Task&nbsp;2.10, you can still work on this task by using a pretrained GPT-2 model from [Hugging Face](https://huggingface.co/openai-community/gpt2). The next code cell shows how you would instantiate this model. Note that you may have to first install the `transformers` library.

In [ ]:
# from transformers import GPT2LMHeadModel
# model = GPT2LMHeadModel.from_pretrained("gpt2")
# logits = model(context).logits[:, -1, :]

#### 🎓 Task 2.11: Sampling-based text generation

 The greedy approach to text generation is not very interesting for practical applications because it always chooses the most likely token, leading to predictable and less creative results. Your task is to modify the code for the `generate()` function to use a **sampling-based approach** instead. In this approach, the next token is chosen randomly based on the probabilities assigned by the model (softmax-normalised logits), treating them as a categorical distribution over the token vocabulary. Additionally, your code should include two common techniques to improve sampling:
 
 * **temperature scaling**, which lets the user control the randomness of the sampling
 * **top-$k$ sampling**, which limits the sampling to the top-$k$ most likely tokens, ignoring less probable ones


 ANSWER: Check the code snippet above.

### Evaluating the pretrained model

If you have experimented with your pretrained GPT-2 model, you will have noticed that its ability to generate useful text is somewhat limited. By today’s standards, GPT-2 is a small model with modest capabilities. However, it can still be helpful for certain tasks, such as text autocompletion, generating filler text, or answering simple questions. To rigourosly evaluate language models, researchers often use standard benchmark datasets. Creating these benchmarks is a discipline of its own, and they tend to become increasingly challenging as models continue to improve.

In the final task of this lab, you will evaluate GPT-2’s performance on a small subset of the [HellaSwag dataset](https://rowanzellers.com/hellaswag/), which was published in the same year as GPT-2 itself (2019). HellaSwag is designed to test a model’s ability to perform commonsense reasoning in challenging contexts. Unlike simpler benchmarks, HellaSwag presents scenarios where the correct text completion depends on semantic relationships between events and on world knowledge. This makes it a good choice for assessing the ability of language models to go beyond surface-level patterns and produce meaningful, context-aware predictions.

#### 🎓 Task 2.12: Evaluating the pretrained model

Read the [HellaSwag website](https://rowanzellers.com/hellaswag/) to get some background on the benchmark. How does a sample from the dataset look like? What is an expected prediction? How does the benchmark allow us to score models? What is the random baseline? What is the human performance reported on the task?

The next cell contains code for evaluating your pretrained model on a small sample from HellaSwag. You will also need a tokenizer. The HellaSwag subset is in the file `hellaswag-mini.jsonl`. Inspect that file to understand the format. Next, read the code and explain how it works. Specifically, how does the code compute the score of individual endings? In the call to `cross_entropy()`, why are the tensors sliced in this specific way?

Finally, what overall score does the pretrained GPT-2 model get on this benchmark? How does that score compare to the random baseline and the human performance?

the is based on some general knowledge and should be selected. 
![alt text](2026-02-13_01-06.png)


Because the model uses Adversarial Filtering, the expected predictions are not be the most perfect ones. 
The random baseline'= 25%                    The human performance = 96.6%

Computing the score of individual endigs: The model feeds the calculate the cross-entropy for the endings. where a lower score means the ending is more likeley (less surprised) and a high score means the ending is unlikely. 

The slicing part  ending_score = torch.nn.functional.cross_entropy(
                    logits[0, -len(suffix) - 1 : -1], context[0, -len(suffix) :]
                ) is used to align the models predictions with the actual target words for the suffix. By shifting the logits slice back by one position relative to target slice, we ensure that the prediction for each word in the suffix comes from the correct privios word in the sequece.


We got a score of  around 36% wich is a bit above the randome performance but far awway from the human performance. 

In [ ]:
import json

with open("hellaswag-mini.jsonl") as f:
    n_correct = 0
    n_total = 0
    for line in f:
        sample = json.loads(line)
        prefix = tokenizer.encode(sample["ctx"]) 
        ending_scores = []
        for i, ending in enumerate(sample["endings"]):
            suffix = tokenizer.encode(" " + ending)
            context = torch.tensor([prefix + suffix], dtype=torch.long)
            with torch.no_grad():
                logits = model(context)
                ending_score = torch.nn.functional.cross_entropy(
                    logits[0, -len(suffix) - 1 : -1], context[0, -len(suffix) :]
                )
            ending_scores.append((ending_score, i))
        predicted = min(ending_scores)[1]
        n_correct += int(predicted == sample["label"])
        n_total += 1
    print(f"Accuracy: {n_correct / n_total:.2%}")

**🥳 Congratulations on finishing lab&nbsp;2!**